# Anima: text readers read descriptions that name no side of the group and should leave the sliders at zero

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/anima-trainer/blob/main/notebooks/anima_colab_offtarget_readers.ipynb)

A reader wired into a picture system meets many descriptions that name none of its sliders' sides. Here 32 descriptions name
the three attributes without any value of the group's dimensions (long or wavy hair, a composed look or a blank expression,
small or close-set eyes), on the same characters and seeds as before; the right amount is zero on every slider. Six readers,
each with its own reading fitted earlier on the 96 training descriptions: Qwen3 and Beatrix reading the plain description,
Beatrix and the framed nano tap reading it in the framed grammar, and two untrained controls in the framed grammar. The
readings were made beforehand and live in the data repo's `beatrix/cgo_readers.json`.

**Nothing is drawn twice.** A picture depends only on its character, seed and the three amounts, so a description read at
zero on every slider is the plain picture itself (copied), and any picture already drawn is copied too; the plain prompt's
pictures and the group at amount 1 come from
[AbstractPhil/geolip-beatrix-anima-data](https://huggingface.co/datasets/AbstractPhil/geolip-beatrix-anima-data); each reader's
set is saved there under `sets/cgo2_*` as soon as it is drawn.

**Content screen.** As before: a picture rated questionable or explicit, or read as child, loli or aged down above 0.2, is
left off the sheets and its set is not uploaded; the fourth character's pictures carry "child, loli, aged down" in the
negative prompt.

**The rule (fixed before the run).** For each reader and pair: a pair stays plain when at least 24 of the 32 amounts round to
zero and the pictures move on that pair, on average, less than a quarter of the group's full move; a reader stays plain on the
group when all three pairs do. Beatrix in the framed grammar is credited against her untrained copy, the framed nano tap
against its untrained body: a pair is credited when the reader stays plain on it and its control does not.

**Cost.** At most 192 pictures at 768 px and 30 steps (a description read at zero everywhere is copied); well under an hour
on the RTX PRO 6000. Results: `experiments/e058_anima_offtarget_readers` of
[AbstractPhil/geolip-beatrix-anima](https://huggingface.co/AbstractPhil/geolip-beatrix-anima).


## 0 · Confirm the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap: the runner at its release, and the dependencies (restart ONCE the first time)

Clones the runner at the release this notebook was written for, and the diffusion-pipe fork beside it (its Anima model
code draws the pictures), then installs the dependencies. **Run the cell; if the kernel restarts, run it again** (it
then skips the install).

In [ ]:
import os, sys, subprocess
TAG = "v0.10.0"                       # this notebook's release of the runner
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "--tags", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", TAG], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the diffusion-pipe fork
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · Set up

Checks the GPU, finds the fork and downloads the three model files. Needs a **write** `HF_TOKEN` in Colab Secrets: the
results and the drawn sets are uploaded.

In [ ]:
from geolip_anima_trainer.anima_runner import AnimaRunner
a = AnimaRunner()      # Anima-Base v1.0 at 768 px; results -> AbstractPhil/geolip-beatrix-anima, sets -> ...-anima-data
a.setup()              # env + HF login + GPU check + the fork + the three model files (~5.6 GB)

## 3 · The readers on the off-target descriptions

Downloads the readers file, prints the job's size (which sets come from the data repo, how many distinct pictures it
draws), then time spent and time left with one line per set, and at the end one line per reader: the verdict, each pair's
amounts at zero and the pictures' mean move on it, and how many pictures were left plain. A run already done is skipped
(`force=True` reruns the reads; the sets come back from the data repo).


In [ ]:
from huggingface_hub import hf_hub_download
readers = hf_hub_download("AbstractPhil/geolip-beatrix-anima-data", "beatrix/cgo_readers.json", repo_type="dataset")
meta = a.run_group_readers(readers, test="e058")     # the off-target tier: 6 readers x 32 descriptions


## Notes

- **Where things land:** `experiments/e058_anima_offtarget_readers/` in the experiments repo (README with the question,
  the design, the rule fixed before the run and the result; `meta.json`; `readers.json`; `result.json` with every picture's
  score on every pair; `tagger_probs.npz`; `sheet_off.jpg`). The sets:
  `sets/cgo2_<reader>_off-<key>/` in the data repo. Locally: `/content/anima_data/`.
- **After a disconnect:** run steps 1-3 again. Finished sets come back from the data repo; only the unfinished set is
  drawn again.
- **The pictures** are outputs of Anima-Base v1.0 (CircleStone Labs Non-Commercial License: outputs are not derivatives
  of the model; the restriction applies to the model and its derivatives, such as LoRAs).
